<a href="https://colab.research.google.com/github/kollikrishnarao-star/Rice-disease-Advisory/blob/main/Copy_of_ver2_online_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# -----------------------------
# CONFIG
# -----------------------------
OUTPUT_FILE = "pest_5lag_dataset_fixed_log_ver2.csv"
MIN_ACTIVE = 2


# -----------------------------
# UPLOAD CSV FROM USER
# -----------------------------
print("Upload training dataset CSV")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df.columns = df.columns.str.strip()


# -----------------------------
# FILTER COLLECTION TYPE
# -----------------------------
VALID_TYPES = ["Number/Light trap", "Percentage"]
df = df[df["Collection Type"].isin(VALID_TYPES)]


# -----------------------------
# GLOBAL SORT
# -----------------------------
df = df.sort_values([
    "PEST NAME",
    "Location",
    "Observation Year",
    "Standard Week"
])


# -----------------------------
# BUILD 5-LAG DATASET
# -----------------------------
rows = []

group_cols = ["PEST NAME", "Location"]

for (pest, loc), g in df.groupby(group_cols):

    g = g.reset_index(drop=True)

    pest_vals = g["Pest Value"].values

    # Need at least 6 weeks
    if len(g) < 6:
        continue

    for i in range(5, len(g) - 1):

        p_now  = max(pest_vals[i], 0)
        p_next = max(pest_vals[i+1], 0)

        p_lag1 = max(pest_vals[i-1], 0)
        p_lag2 = max(pest_vals[i-2], 0)
        p_lag3 = max(pest_vals[i-3], 0)
        p_lag4 = max(pest_vals[i-4], 0)
        p_lag5 = max(pest_vals[i-5], 0)

        # Skip inactive window
        if (
            p_now  < MIN_ACTIVE and
            p_next < MIN_ACTIVE and
            p_lag1 < MIN_ACTIVE and
            p_lag2 < MIN_ACTIVE and
            p_lag3 < MIN_ACTIVE and
            p_lag4 < MIN_ACTIVE and
            p_lag5 < MIN_ACTIVE
        ):
            continue

        row = {
            "pest": pest,
            "location": loc,
            "year": g.loc[i, "Observation Year"],
            "week": g.loc[i, "Standard Week"],

            # Lags
            "pest_t": p_now,
            "pest_lag1": p_lag1,
            "pest_lag2": p_lag2,
            "pest_lag3": p_lag3,
            "pest_lag4": p_lag4,
            "pest_lag5": p_lag5,

            # Climate
            "MaxT": g.loc[i, "MaxT"],
            "MinT": g.loc[i, "MinT"],
            "RH1": g.loc[i, "RH1(%)"],
            "RH2": g.loc[i, "RH2(%)"],
            "RF": g.loc[i, "RF(mm)"],
            "WS": g.loc[i, "WS(kmph)"],
            "SSH": g.loc[i, "SSH(hrs)"],
            "EVP": g.loc[i, "EVP(mm)"],

            # Target
            "pest_next": p_next,
            "delta_pest": p_next - p_now
        }

        rows.append(row)


# -----------------------------
# FINALIZE
# -----------------------------
final_df = pd.DataFrame(rows)

PEST_COLS = [
    "pest_t",
    "pest_lag1",
    "pest_lag2",
    "pest_lag3",
    "pest_lag4",
    "pest_lag5",
    "pest_next"
]

NO_LOG_PESTS = ["LeafBlast", "NeckBlast"]


def conditional_log(pest, value):

    if pest in NO_LOG_PESTS:
        return value

    return np.log1p(value)


for col in PEST_COLS:

    final_df[col] = final_df.apply(
        lambda r: conditional_log(r["pest"], r[col]),
        axis=1
    )


# -----------------------------
# SAVE
# -----------------------------
final_df.to_csv(OUTPUT_FILE, index=False)


# -----------------------------
# DOWNLOAD RESULT
# -----------------------------
files.download(OUTPUT_FILE)


# -----------------------------
# REPORT
# -----------------------------
print("Fixed 5-lag dataset created")
print("Rows:", len(final_df))
print("Saved to:", OUTPUT_FILE)

print("\nCheck pest scales:")
print(final_df[PEST_COLS].describe())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# CONFIG
# -----------------------------
MI_THRESHOLD = 0.3

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("pest_5lag_dataset_fixed_log_ver2.csv")

climate_vars = ['MaxT','MinT','RH1','RH2','RF','WS','SSH','EVP','week']


# -----------------------------
# MI FUNCTION
# -----------------------------
def compute_avg_mi(X, y):

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)

    y = y.loc[X.index]

    mi = mutual_info_regression(X, y, random_state=42)

    return np.mean(mi)


# -----------------------------
# FEATURE SETS
# -----------------------------
def build_feature_sets(df):

    return {
        "Climate only": df[climate_vars],
        "Climate + Lag1": df[climate_vars + ['pest_t']],
        "Climate + Lag2": df[climate_vars + ['pest_t','pest_lag1']],
        "Climate + Lag3": df[climate_vars + ['pest_t','pest_lag1','pest_lag2']],
        "Climate + Lag4": df[climate_vars + ['pest_t','pest_lag1','pest_lag2','pest_lag3']],
        "Climate + Lag5": df[climate_vars + ['pest_t','pest_lag1','pest_lag2','pest_lag3','pest_lag4']]
    }


# -----------------------------
# RUN MI AGENT
# -----------------------------
all_results = []

selected_pests = []
rejected_pests = []

for pest in df['pest'].unique():

    df_p = df[df['pest'] == pest]

    if len(df_p) < 100:
        continue

    y = df_p['pest_next']
    feature_sets = build_feature_sets(df_p)

    results = {}

    for name, X in feature_sets.items():
        score = compute_avg_mi(X, y)
        results[name] = score

    best = max(results, key=results.get)
    max_mi = results[best]

    # -----------------------------
    # SELECTION LOGIC
    # -----------------------------
    if max_mi >= MI_THRESHOLD:
        status = "Selected for Training"
        selected_pests.append(pest)
    else:
        status = "Rejected"
        rejected_pests.append(pest)

    all_results.append({
        "pest": pest,
        "Climate_only": results["Climate only"],
        "Lag1": results["Climate + Lag1"],
        "Lag2": results["Climate + Lag2"],
        "Lag3": results["Climate + Lag3"],
        "Lag4": results["Climate + Lag4"],
        "Lag5": results["Climate + Lag5"],
        "Best_MI": max_mi,
        "Agent_Selected": best,
        "Training_Status": status
    })

    # ---- Plot ----
    plt.figure()
    plt.bar(results.keys(), results.values())
    plt.axhline(MI_THRESHOLD, color='red', linestyle='--', label='Threshold')
    plt.title(f"MI Comparison - {pest}")
    plt.ylabel("Avg MI")
    plt.xticks(rotation=20)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{pest}_MI_plot_5lag.png")
    plt.show()


# -----------------------------
# SUMMARY TABLE
# -----------------------------
mi_table = pd.DataFrame(all_results)
mi_table = mi_table.sort_values("pest")

print("\nFull MI Table:")
print(mi_table)

mi_table.to_csv("MI_agent_decisions_5lag_log.csv", index=False)


# -----------------------------
# SHOW SELECTED VS REJECTED
# -----------------------------
print("\n============================")
print("Pests Selected For Training")
print("============================")

for p in selected_pests:
    print(p)

print("\n============================")
print("Pests Rejected")
print("============================")

for p in rejected_pests:
    print(p)


# Save lists
pd.DataFrame({"Selected_Pests": selected_pests}).to_csv("selected_pests.csv", index=False)
pd.DataFrame({"Rejected_Pests": rejected_pests}).to_csv("rejected_pests.csv", index=False)


# -----------------------------
# HEATMAP
# -----------------------------
plt.figure(figsize=(10,6))

sns.heatmap(
    mi_table.set_index("pest")[["Climate_only","Lag1","Lag2","Lag3","Lag4","Lag5"]],
    annot=True,
    cmap="YlGnBu"
)

plt.title("Mutual Information Strength by Feature Set (Lag5)")
plt.tight_layout()
plt.savefig("MI_heatmap_5lag.png")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

mi = pd.read_csv("MI_agent_decisions_5lag_log.csv")
plt.figure(figsize=(12,7))

lags = ['Climate_only','Lag1','Lag2','Lag3','Lag4','Lag5']

for _, row in mi.iterrows():

    values = [0] + [row[l] for l in lags]   # start from 0
    x = list(range(len(values)))

    plt.plot(x, values, marker='o', label=row['pest'])

plt.xticks(
    ticks=range(7),
    labels=['0','Climate','Lag1','Lag2','Lag3','Lag4','Lag5']
)

plt.ylabel("Average Mutual Information")
plt.title("Pest Memory Growth Across Lag Depth")
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
plt.grid(True)

plt.tight_layout()
plt.savefig("MI_Lag_Curves_AllPests.png")
plt.show()


In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder


# -----------------------------
# LOAD DATA
# -----------------------------

df = pd.read_csv("pest_5lag_dataset_fixed_log.csv")

df["MaxT"] = df["MaxT"].clip(15, 45)
df["MinT"] = df["MinT"].clip(5, 30)

df["RH1"] = df["RH1"].clip(20, 100)
df["RH2"] = df["RH2"].clip(10, 100)

df["RF"] = df["RF"].clip(0, 350)


df["SSH"] = df["SSH"].clip(0, 84)

# Drop very rare pest
#df = df[df["pest"] != "NeckBlast"]


# -----------------------------
# FEATURES & TARGET
# -----------------------------
climate = ['MaxT','MinT','RH1','RH2','RF','WS','SSH','EVP','week']


def run_model(X, y, pest, mode):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=10
    )

    model = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=10,
        objective="reg:squarederror"
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    NO_LOG = ["LeafBlast", "NeckBlast"]

    if pest in NO_LOG:
        y_test_real = y_test
        preds_real = preds
    else:
        y_test_real = np.expm1(y_test)
        preds_real = np.expm1(preds)

    preds_real = np.clip(preds_real, 0, None)



    rmse_real = np.sqrt(mean_squared_error(y_test_real, preds_real))
    r2_real = r2_score(y_test_real, preds_real)

    # -----------------------------
    # SAVE TEST PREDICTIONS
    # -----------------------------
    out = X_test.copy()
    out["y_test"] = y_test_real
    out["preds"] = preds_real

    out.to_csv(f"test_preds_{pest}_{mode}.csv", index=False)

    return rmse_real, r2_real




# -----------------------------
# ENCODE LOCATION
# -----------------------------

le_loc = LabelEncoder()
df["location_enc"] = le_loc.fit_transform(df["location"])


def get_features(df, mode):

    if mode == "climate":
        return df[climate]

    if mode == "lag1":
        return df[climate + ['pest_t']]

    if mode == "lag2":
        return df[climate + ['pest_t','pest_lag1']]

    if mode == "lag3":
        return df[climate + ['pest_t','pest_lag1','pest_lag2']]

    if mode == "lag4":
        return df[climate + ['pest_t','pest_lag1','pest_lag2','pest_lag3']]

    if mode == "lag5":
        return df[climate + ['pest_t','pest_lag1','pest_lag2','pest_lag3','pest_lag4']]


results = []

for pest in df['pest'].unique():

    d = df[df['pest']==pest]
    d = d.sort_values(['year','week'])

    if len(d) < 100:
        continue

    y = d['pest_next']

    r = {"pest": pest}

    for mode in ['climate','lag1','lag2','lag3','lag4','lag5']:

        X = get_features(d, mode)

        rmse, r2 = run_model(X, y, pest, mode)

        r[f"RMSE_{mode}"] = rmse
        r[f"R2_{mode}"] = r2


    results.append(r)

perf_table = pd.DataFrame(results)

perf_table.to_csv("Lag_Performance_Comparison_5lag_log_real.csv", index=False)






In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

perf = pd.read_csv("Lag_Performance_Comparison_5lag_log_real.csv")
mi = pd.read_csv("MI_agent_decisions_5lag_log.csv")

lag_map = {
    "Climate only":0,
    "Climate + Lag1":1,
    "Climate + Lag2":2,
    "Climate + Lag3":3,
    "Climate + Lag4":4,
    "Climate + Lag5":5
}

labels = ['Climate','Lag1','Lag2','Lag3','Lag4','Lag5']
x = list(range(6))

for pest in perf['pest']:

    p_perf = perf[perf['pest']==pest].iloc[0]
    p_mi = mi[mi['pest']==pest].iloc[0]

    r2_vals = [
        p_perf['R2_climate'],
        p_perf['R2_lag1'],
        p_perf['R2_lag2'],
        p_perf['R2_lag3'],
        p_perf['R2_lag4'],
        p_perf['R2_lag5']
    ]

    rmse_vals = [
        p_perf['RMSE_climate'],
        p_perf['RMSE_lag1'],
        p_perf['RMSE_lag2'],
        p_perf['RMSE_lag3'],
        p_perf['RMSE_lag4'],
        p_perf['RMSE_lag5']
    ]

    mi_choice = p_mi['Agent_Selected']
    mi_idx = lag_map[mi_choice]

    climate_idx = 0
    best_r2_idx = r2_vals.index(max(r2_vals))
    best_rmse_idx = rmse_vals.index(min(rmse_vals))

    # ------------------- R2 Plot -------------------
    plt.figure()
    plt.plot(x, r2_vals, marker='o')

    # Climate baseline (RED hollow)
    plt.scatter(climate_idx, r2_vals[climate_idx],
                s=200, facecolors='none', edgecolors='red', linewidths=2.5,label="Only Climate")

    # Best model (GREEN hollow)
    plt.scatter(best_r2_idx, r2_vals[best_r2_idx],
                s=200, facecolors='none', edgecolors='orange', linewidths=2.5,label="Best R2")

    # MI selected (BLUE hollow)
    plt.scatter(mi_idx, r2_vals[mi_idx],
                s=200, facecolors='none', edgecolors='black', linewidths=2.5,label="Agent Suggested")

    plt.xticks(x, labels)
    plt.title(f"R2 vs Lag - {pest}")
    min_r2 = min(r2_vals)
    plt.ylim(min_r2, 1)
    plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x*100:.0f}"))
    plt.ylabel("R2")
    plt.legend(markerscale=0.6)
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{pest}_R2_MI.png")
    plt.show()

    # ------------------- RMSE Plot -------------------
    plt.figure()
    plt.plot(x, rmse_vals, marker='s')

    plt.scatter(climate_idx, rmse_vals[climate_idx],
                s=200, facecolors='none', edgecolors='red', linewidths=2.5, label="Only Climate")

    plt.scatter(best_rmse_idx, rmse_vals[best_rmse_idx],
                s=200, facecolors='none', edgecolors='orange', linewidths=2.5, label="Best RMSE")

    plt.scatter(mi_idx, rmse_vals[mi_idx],
                s=200, facecolors='none', edgecolors='black', linewidths=2.5, label="Agent Suggested")

    plt.xticks(x, labels)
    plt.title(f"RMSE vs Lag - {pest}")

    min_rmse = min(rmse_vals)
    max_rmse = max(rmse_vals)

    span = max_rmse - min_rmse
    plt.ylim(min_rmse - 0.5*span, max_rmse + 0.5*span)

    plt.ylabel("RMSE")
    plt.legend(markerscale=0.6)
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{pest}_RMSE_MI.png")
    plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

perf = pd.read_csv("Lag_Performance_Comparison_5lag_log_real.csv")
mi = pd.read_csv("MI_agent_decisions_5lag_log.csv")

rmse_map = {
    "Climate only":"RMSE_climate",
    "Climate + Lag1":"RMSE_lag1",
    "Climate + Lag2":"RMSE_lag2",
    "Climate + Lag3":"RMSE_lag3",
    "Climate + Lag4":"RMSE_lag4",
    "Climate + Lag5":"RMSE_lag5"
}

r2_map = {
    "Climate only":"R2_climate",
    "Climate + Lag1":"R2_lag1",
    "Climate + Lag2":"R2_lag2",
    "Climate + Lag3":"R2_lag3",
    "Climate + Lag4":"R2_lag4",
    "Climate + Lag5":"R2_lag5"
}

results = []

for pest in perf['pest']:

    p_perf = perf[perf['pest']==pest].iloc[0]
    p_mi = mi[mi['pest']==pest].iloc[0]

    mi_lag = p_mi['Agent_Selected']

    # RMSE Improvement (%)
    climate_rmse = p_perf['RMSE_climate']
    mi_rmse = p_perf[rmse_map[mi_lag]]
    rmse_imp = ((climate_rmse - mi_rmse) / climate_rmse) * 100

    # R2 Gain (Absolute)
    climate_r2 = p_perf['R2_climate']
    mi_r2 = p_perf[r2_map[mi_lag]]
    r2_gain = mi_r2 - climate_r2

    results.append({
        "pest": pest,
        "RMSE_Improvement_%": rmse_imp,
        "R2_Gain": r2_gain
    })

imp_df = pd.DataFrame(results)


plt.figure(figsize=(10,5))

plt.bar(imp_df['pest'], imp_df['RMSE_Improvement_%'])

plt.axhline(0)
plt.xticks(rotation=45)
plt.ylabel("% RMSE Improvement")
plt.title("RMSE Improvement by MI Agent over pure climate model")

plt.tight_layout()
plt.savefig("RMSE_Improvement.png")
plt.show()

plt.figure(figsize=(10,5))

plt.bar(imp_df['pest'], imp_df['R2_Gain'])

plt.axhline(0)
plt.xticks(rotation=45)
plt.ylabel("Δ R² Gain")
plt.title("R² Gain by MI Agent")

plt.tight_layout()
plt.savefig("R2_Gain.png")
plt.show()


In [ ]:
!pip install ipywidgets
